# Single-Cell Analysis  – May 2026

End-to-end preprocessing pipeline for paired **GEX + TCR + ADT** data from
sorted CD8 TIL samples. 

**Outline**
1. Imports
2. Sample info
3. Data loading (GEX + TCR per sample)
4. MuData assembly
5. Gene-ID annotation (mouse, Ensembl)
6. ADT modality extraction
7. Filtering genes
8. Quality-control metrics
9. Normalisation & HVG selection
10. Dimensionality reduction & clustering
11. Metadata annotation
12. AIRR-based subsetting & final save
13. Save mdata

## 1 · Imports

In [1]:
# Standard library
import os
from pathlib import Path
from functools import partial

# Numerical / data
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation

# Single-cell ecosystem
import anndata as ad
import mudata as mu
import scanpy as sc
import scirpy as ir

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt

# Optional / downstream
import decoupler as dc


alt.data_transformers.enable('vegafusion')

sc.settings.verbosity = 1 

## 2 · Sample info

Define all samples with their experimental group and treatment condition.
TIL (tumour-infiltrating lymphocyte) 2021 cohort.

In [2]:
samples = {
    "10mix-ICI1": {"group": "10mix", "condition":"ICI"},
    "10mix-ICI2": {"group": "10mix","condition":"ICI"},
    "11mix-ICI1": {"group": "11mix","condition":"ICI"},
    "11mix-ICI2": {"group": "11mix","condition":"ICI"},
    "GF-ICI1": {"group": "GF","condition":"No ICI"},
    "GF-ICI2": {"group": "GF","condition":"No ICI"},
    "GF-ICI1-plus": {"group": "GF-plus","condition":"ICI"},
    "GF-ICI2-plus": {"group": "GF-plus","condition":"ICI"},
    "10mix1": {"group": "10mix", "condition":"naive"},
    "10mix2": {"group": "10mix", "condition":"naive"},
    "11mix1": {"group": "11mix", "condition":"naive"},
    "11mix2": {"group": "11mix", "condition":"naive"},
    "GF1": {"group": "GF", "condition":"naive"},
    "GF2": {"group": "GF", "condition":"naive"},
}    

## 3 · Data Loading

Read one **GEX** (gene expression) and one **TCR** (VDJ) AnnData per sample
from the CellRanger multi output directories.

In [3]:
samples = {
    "10mix-ICI1": {"group": "10mix", "condition":"ICI"},
    "10mix-ICI2": {"group": "10mix","condition":"ICI"},
    "11mix-ICI1": {"group": "11mix","condition":"ICI"},
    "11mix-ICI2": {"group": "11mix","condition":"ICI"},
    "GF-ICI1": {"group": "GF","condition":"No ICI"},
    "GF-ICI2": {"group": "GF","condition":"No ICI"},
    "GF-ICI1-plus": {"group": "GF-plus","condition":"ICI"},
    "GF-ICI2-plus": {"group": "GF-plus","condition":"ICI"},


}    
    
# Create a list of AnnData objects (one for each sample)
adatas_tcr_2021 = {}
adatas_gex_2021 = {}
for sample, sample_meta in samples.items():
    adata_gex_2021 = sc.read_10x_h5(f"/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/2021-02-01_sorted_cd8_til/analyses_icbi/{sample}/outs/per_sample_outs/{sample}/count/sample_filtered_feature_bc_matrix.h5", gex_only=False)
    #adata_tcr_2021 = ir.io.read_10x_vdj(f"/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/2021-02-01_sorted_cd8_til/analyses_icbi/{sample}/outs/per_sample_outs/{sample}/vdj_t/filtered_contig_annotations.csv")
    adata_tcr_2021 = ir.io.read_10x_vdj(f"/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/2021-02-01_sorted_cd8_til/analyses_icbi/{sample}/outs/multi/vdj_t/all_contig_annotations.csv")
    # concatenation only works with unique gene names
    adata_gex_2021.var_names_make_unique()
    adatas_tcr_2021[sample] = adata_tcr_2021
    adatas_gex_2021[sample] = adata_gex_2021

/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/airr/schema.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
Non-standard locus name: None 
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/anndata/utils.py:354: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(msg, category, stacklevel=stacklevel)
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/anndata/_co

### Concatenate per-sample AnnData objects

In [4]:
adata_tcr_2021 = ad.concat(adatas_tcr_2021, index_unique="_")

In [5]:
adata_gex_2021 = ad.concat(adatas_gex_2021, index_unique="_")

## 4 · MuData Assembly

Combine GEX and TCR (AIRR) into a single `MuData` container.

In [6]:
mdata = mu.MuData({"gex": adata_gex_2021, "airr": adata_tcr_2021})

/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## 5 · Gene-ID Annotation

Map gene symbols → Ensembl IDs (mouse / mm10) using **mygene**.
This is required for downstream tools that expect Ensembl IDs.

In [7]:
import mygene
import pandas as pd

mg = mygene.MyGeneInfo()

genes = mdata["gex"].var_names.tolist()

res = mg.querymany(
    genes,
    scopes="symbol",
    fields="ensembl.gene",
    species="mouse",
    as_dataframe=True
)

res = res.reset_index().rename(columns={"query": "gene_symbol"})

res.columns

2026-05-28 13:18:04 | [WARNING] Input sequence provided is already in string format. No operation performed
2026-05-28 13:18:04 | [WARNING] Input sequence provided is already in string format. No operation performed
2026-05-28 13:18:04 | [INFO] querying 1-1000 ...
2026-05-28 13:18:06 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-05-28 13:18:07 | [INFO] querying 1001-2000 ...
2026-05-28 13:18:08 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-05-28 13:18:09 | [INFO] querying 2001-3000 ...
2026-05-28 13:18:11 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-05-28 13:18:12 | [INFO] querying 3001-4000 ...
2026-05-28 13:18:13 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-05-28 13:18:14 | [INFO] querying 4001-5000 ...
2026-05-28 13:18:15 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-05-28 13:18:16 | [INFO] querying 5001-6000

Index(['gene_symbol', '_id', '_score', 'ensembl.gene', 'notfound', 'ensembl'], dtype='object')

In [8]:
# Case 1: if column is called "ensembl.gene"
if "ensembl.gene" in res.columns:
    mapping = (
        res.dropna(subset=["ensembl.gene"])
        .drop_duplicates("gene_symbol")
        .set_index("gene_symbol")["ensembl.gene"]
    )

# Case 2: if column is called "ensembl"
elif "ensembl" in res.columns:
    def extract_ensembl(x):
        if isinstance(x, dict):
            return x.get("gene")
        elif isinstance(x, list):
            return x[0].get("gene") if len(x) > 0 else None
        else:
            return None

    res["gene_ids"] = res["ensembl"].apply(extract_ensembl)

    mapping = (
        res.dropna(subset=["gene_ids"])
        .drop_duplicates("gene_symbol")
        .set_index("gene_symbol")["gene_ids"]
    )

In [9]:
mdata["gex"].var["gene_ids"] = mdata["gex"].var_names.map(mapping)
mdata["gex"].var["feature_types"] = "Gene Expression"

## 6 · ADT Modality Extraction

Split antibody-capture (ADT) features into a dedicated `mdata['adt']` modality.

In [10]:
mask = (
    mdata["gex"].var["gene_ids"].isna() &
    mdata["gex"].var.index.str.contains("_TotalSeqC")
)

mdata["gex"].var.loc[mask, "feature_types"] = "Antibody Capture"

In [11]:
mdata.mod["adt"] = mdata["gex"][
    :,
    mdata["gex"].var["feature_types"] == "Antibody Capture"
].copy()

In [12]:
mdata.update()

/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:931: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var meth

## 7 · Filtering genes

In [13]:
# filter genes in gex modality
sc.pp.filter_genes(mdata["gex"], min_cells=10)

## 8 · Cell-Level Metadata Annotation

Extract `sample_id` from cell barcodes, then derive:
- `origin` – tissue of origin (TIL vs colon)
- `batch_id` – sequencing batch
- `condition` – experimental condition per sample
- `treatment` – ICI / no_ICI
- `received_ici` – boolean flag (Yes / No)
- `year` / `tissue` – study metadata

In [14]:
def update_columns_origin(row):

    if "ICI" not in row["sample_id"]:
        row["origin"] = "colon"
    else:
        row["origin"] = "til"

    return row

In [15]:
def update_columns_batch(row):

    if "ICI1" in row["sample_id"]:
        row["batch_id"] = "ICI1"
    elif "ICI2" in row["sample_id"]:
        row["batch_id"] = "ICI2"
    elif row["sample_id"] == "10mix1":
        row["batch_id"] = "1"
    elif row["sample_id"] == "10mix2":
        row["batch_id"] = "2"

    return row

In [16]:
def update_columns_condition(row):
    
    if row["sample_id"] == "GF-ICI2-plus":
        row["condition"] = "GF-plus"
    elif row["sample_id"] == "GF-ICI1-plus":
        row["condition"] = "GF-plus"
    elif row["sample_id"] == "GF-ICI2":
        row["condition"] = "GF"
    elif row["sample_id"] == "GF-ICI1":
        row["condition"] = "GF"
    elif row["sample_id"] == "10mix-ICI1":
        row["condition"] = "10mix"
    elif row["sample_id"] == "10mix-ICI2":
        row["condition"] = "10mix"
    elif row["sample_id"] == "11mix-ICI1":
        row["condition"] = "11mix"
    elif row["sample_id"] == "11mix-ICI2":
        row["condition"] = "11mix"

    return row

In [17]:
mdata["gex"].obs["sample_id"] = (
    mdata["gex"].obs_names
    .str.split("_")
    .str[-1]
)

In [18]:
mdata["gex"].obs = mdata["gex"].obs.apply(update_columns_origin, axis=1)

In [19]:
mdata["gex"].obs = mdata["gex"].obs.apply(update_columns_batch, axis=1)

In [20]:
mdata["gex"].obs = mdata["gex"].obs.apply(update_columns_condition, axis=1)

In [21]:
mdata.update()

/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:931: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var meth

## 9 · Quality-Control Metrics

- Counts layer contains raw counts

Flag mitochondrial, ribosomal, and haemoglobin genes,
then compute per-cell QC statistics.

In [22]:
mdata["gex"].layers["counts"] = mdata["gex"].X.copy()

In [23]:
mdata["gex"].var["mito"] = mdata["gex"].var_names.str.startswith("mt")
mdata["gex"].var["ribo"] = mdata["gex"].var_names.str.startswith("rb")
mdata["gex"].var["hb"] = mdata["gex"].var_names.str.startswith("hb")
sc.pp.calculate_qc_metrics(
    mdata["gex"],
    qc_vars=["mito", "ribo", "hb"],
    inplace=True,
    percent_top=[20],
    log1p=True,
)

## 10 · Normalisation & Highly Variable Genes

Normalise library size, log-transform, and select the top 1000 highly_variable_genes

In [24]:
sc.pp.normalize_total(mdata["gex"])
sc.pp.log1p(mdata["gex"])

In [25]:
sc.pp.highly_variable_genes(mdata["gex"], n_top_genes=1000, batch_key="sample_id")

... storing 'sample_id' as categorical
... storing 'origin' as categorical
... storing 'batch_id' as categorical
... storing 'condition' as categorical
... storing 'gene_ids' as categorical
... storing 'feature_types' as categorical


## 11 · Dimensionality Reduction & Leiden Clustering

Build a kNN graph, embed with UMAP, and compute Leiden clusters at three
resolutions for exploratory inspection.

In [26]:
sc.pp.neighbors(mdata["gex"])

/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/scanpy/neighbors/__init__.py:586: UserWarning: You’re trying to run this on 17586 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  X = _choose_representation(self._adata, use_rep=use_rep, n_pcs=n_pcs)
2026-05-28 13:21:28.672569: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-28 13:21:28.733396: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [27]:
sc.tl.umap(mdata["gex"])

In [28]:
sc.tl.leiden(mdata["gex"], resolution=0.4, key_added="rna_leiden_05")

/tmp/ipykernel_3128097/810156193.py:1: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(mdata["gex"], resolution=0.4, key_added="rna_leiden_05")


In [29]:
sc.tl.leiden(mdata["gex"], resolution=1, key_added="rna_leiden")

In [30]:
sc.tl.leiden(mdata["gex"], resolution=0.3, key_added="gex03")

In [31]:
mdata["adt"].obs["sample_id"] = mdata["gex"].obs["sample_id"]

In [32]:
mdata["gex"].obs["year"] = "2021"
mdata["adt"].obs["year"] = "2021"
mdata["adt"].obs["tissue"] = "TIL"
mdata["gex"].obs["tissue"] = "TIL"

In [33]:
treatment_map = {
    "10mix-ICI1": "ICI",
    "10mix-ICI2": "ICI",
    "GF-ICI1-plus": "ICI",
    "GF-ICI2-plus": "ICI",
    "GF-ICI1": "no_ICI",
    "GF-ICI2": "no_ICI",
}

In [34]:
mdata["gex"].obs["treatment"] = (
    mdata["gex"].obs["sample_id"]
    .map(treatment_map)
)

In [35]:
mdata["adt"].obs["treatment"] = mdata["gex"].obs["treatment"]

In [36]:
mdata["gex"].obs["received_ici"] = (
    mdata["gex"].obs["treatment"]
    .eq("ICI")
    .map({True: "Yes", False: "No"})
)

In [37]:
mdata["adt"].obs["received_ici"] = mdata["gex"].obs["received_ici"]

In [38]:
mdata["adt"].obs["sample_id"] = mdata["gex"].obs["sample_id"]

In [39]:
mdata.update()

/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:931: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var meth

## 12 · AIRR-Based Subsetting & Final MuData

Restrict the ADT modality to cells that also have TCR data (AIRR), then
sync the MuData object.

In [40]:
# cells present in AIRR
airr_cells = mdata["airr"].obs_names

# subset ADT to those cells
mdata.mod["adt"] = mdata["adt"][airr_cells, :].copy()

In [41]:
mdata.update()

/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## 13 · Save Mudata


In [46]:
import pandas as pd

for mod in mdata.mod.keys():
    adata = mdata[mod]

    # fix obs columns
    for col in adata.obs.columns:
        if adata.obs[col].dtype == "object":
            adata.obs[col] = adata.obs[col].astype(str)

    # fix var columns
    for col in adata.var.columns:
        if adata.var[col].dtype == "object":
            adata.var[col] = adata.var[col].fillna("").astype(str)

mdata.update()
#mdata.write("001_create_mudata.h5mu")

/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
... storing 'gene_ids' as categorical
/home/kvalem/.conda/envs/scvi_env/lib/python3.10/site-packages/mudata/_c